In [ ]:
import pandas as pd
from nba_api.stats.endpoints import leaguegamelog
import time

def fetch_logs(season='2023-24'):
    print("Descargando datos de partidos...")
    game_logs = leaguegamelog.LeagueGameLog(season=season, season_type_all_star='Regular Season')
    df = game_logs.get_data_frames()[0]
    df = df[['GAME_ID', 'TEAM_ID', 'TEAM_NAME', 'GAME_DATE', 'WL', 'PTS', 'REB']]
    df.columns = ['game_id', 'team_id', 'team_name', 'game_date', 'win', 'points', 'rebounds']
    df['game_date'] = pd.to_datetime(df['game_date'])
    df = df.sort_values(by=['team_name', 'game_date'])
    return df

def compute_team_averages(df):
    print("Calculando estadísticas previas por equipo...")

    df['game_number'] = df.groupby('team_name').cumcount()

    df['avg_points_before'] = df.groupby('team_name')['points'].transform(lambda x: x.shift().expanding().mean())
    df['avg_rebounds_before'] = df.groupby('team_name')['rebounds'].transform(lambda x: x.shift().expanding().mean())

    df = df.dropna(subset=['avg_points_before', 'avg_rebounds_before'])  # eliminar primeros juegos sin data previa

    return df

def build_match_dataset(df):
    print("Unificando datos por partido...")

    games = []
    grouped = df.groupby('game_id')

    for game_id, group in grouped:
        if len(group) == 2:
            team1, team2 = group.iloc[0], group.iloc[1]

            row = {
                'game_id': game_id,
                'home_team': team1['team_name'],
                'away_team': team2['team_name'],
                'home_avg_pts': team1['avg_points_before'],
                'away_avg_pts': team2['avg_points_before'],
                'home_avg_reb': team1['avg_rebounds_before'],
                'away_avg_reb': team2['avg_rebounds_before'],
                'home_win': 1 if team1['win'] == 'W' else 0
            }
            games.append(row)

    df_final = pd.DataFrame(games)
    print(f"Total partidos válidos: {len(df_final)}")
    return df_final

def main():
    raw_logs = fetch_logs()
    with_averages = compute_team_averages(raw_logs)
    final_dataset = build_match_dataset(with_averages)
    final_dataset.to_csv('nba_predictive_lin_dataset.csv', index=False)
    print("Dataset 'nba_predictive_dataset.csv' generado con éxito.")

if __name__ == "__main__":
    main()


Descargando datos de partidos...
Calculando estadísticas previas por equipo...
Unificando datos por partido...
Total partidos válidos: 1215
Dataset 'nba_predictive_dataset.csv' generado con éxito.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [5]:
# Cargar dataset con estadísticas antes de los partidos
df = pd.read_csv("nba_predictive_dataset.csv")

# Calcular promedios generales por equipo
avg_home = df.groupby('home_team')[['home_avg_pts', 'home_avg_reb']].mean()
avg_away = df.groupby('away_team')[['away_avg_pts', 'away_avg_reb']].mean()

# Reemplazar valores por promedios para crear el dataset de entrenamiento
df_option1 = df.copy()
df_option1['home_avg_pts'] = df_option1['home_team'].map(avg_home['home_avg_pts'])
df_option1['home_avg_reb'] = df_option1['home_team'].map(avg_home['home_avg_reb'])
df_option1['away_avg_pts'] = df_option1['away_team'].map(avg_away['away_avg_pts'])
df_option1['away_avg_reb'] = df_option1['away_team'].map(avg_away['away_avg_reb'])


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, classification_report

# Variables predictoras y objetivo
X = df_option1[['home_avg_pts', 'away_avg_pts', 'home_avg_reb', 'away_avg_reb']]
y = df_option1['home_win']  # Variable binaria

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Entrenar modelo de regresión lineal
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluar el modelo
y_pred_continuous = model.predict(X_test)  # Predicciones continuas

y_pred = (y_pred_continuous >= 0.5).astype(int)  # Convertir a binario usando un umbral de 0.5

accuracy = accuracy_score(y_test, (y_pred_continuous >= 0.5).astype(int)) #Accuaracy
confusion_matrix = confusion_matrix(y_test, (y_pred_continuous >= 0.5).astype(int)) #Matriz confusion
print(f"✅ Precisión del modelo: {accuracy:.4f}")
print(classification_report(y_test, y_pred))

# Configurar el gráfico
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico Regresión Lineal
sns.heatmap(confusion_matrix, annot=True, fmt='d', ax=axes[0], cmap='Blues', 
            annot_kws={"size": 16}, cbar=False)
axes[0].set_title('Matriz de Confusión - Regresión Lineal', fontsize=14)
axes[0].set_xlabel('Predicción', fontsize=12)
axes[0].set_ylabel('Real', fontsize=12)
axes[0].set_xticklabels(['Pierde', 'Gana'], fontsize=10)
axes[0].set_yticklabels(['Pierde', 'Gana'], fontsize=10)

plt.tight_layout()
plt.show()

# Corregir la función de predicción
def predecir_partido(home_team, away_team):
    try:
        # Buscar los promedios por equipo
        home = avg_home.loc[home_team]
        away = avg_away.loc[away_team]
    except KeyError as e:
        print(f"❌ Equipo no encontrado: {e}")
        return

    # Armar input para el modelo
    input_data = pd.DataFrame([{
        'home_avg_pts': home['home_avg_pts'],
        'away_avg_pts': away['away_avg_pts'],
        'home_avg_reb': home['home_avg_reb'],
        'away_avg_reb': away['away_avg_reb']
    }])

    # Hacer predicción
    pred_continuous = model.predict(input_data)[0]
    pred = 1 if pred_continuous >= 0.5 else 0  # Convertir a binario usando un umbral de 0.5

    print(f"\n🔮 Predicción para {home_team} vs {away_team}")
    print(f"📊 Valor continuo de predicción: {pred_continuous:.4f}")
    print("✅ Resultado:", "GANA el local" if pred == 1 else "PIERDE el local")

✅ Precisión del modelo: 0.6274
              precision    recall  f1-score   support

           0       0.61      0.50      0.55       167
           1       0.64      0.73      0.68       198

    accuracy                           0.63       365
   macro avg       0.62      0.62      0.62       365
weighted avg       0.63      0.63      0.62       365



In [7]:
# Cambia los equipos según desees probar
predecir_partido("Boston Celtics", "Miami Heat")



🔮 Predicción para Boston Celtics vs Miami Heat
📊 Valor continuo de predicción: 0.7920
✅ Resultado: GANA el local
